# Download DocLayNet

In [11]:
!pip install kagglehub


   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   -------------------- ------------------- 1/2 [kagglehub]
   ---------------------------------------- 2/2 [kagglehub]



In [14]:
import kagglehub
import shutil
import os

# Download dataset (cached by kagglehub)
src_path = kagglehub.dataset_download("dlaproj123/doclaynet")

# Target local directory
dst_path = os.path.join("data", "doclaynet")
os.makedirs(dst_path, exist_ok=True)

print("Copying dataset...")
shutil.copytree(src_path, dst_path, dirs_exist_ok=True)

print("Dataset saved to:", os.path.abspath(dst_path))


Resuming download from 20971520 bytes (29990588955 bytes left)...
Resuming download to C:\Users\posko\.cache\kagglehub\datasets\dlaproj123\doclaynet\1.archive (20971520/30011560475) bytes left.


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 28.0G/28.0G [3:33:49<00:00, 2.34MB/s]

Extracting files...


OSError: [Errno 28] No space left on device

We need to modify the dataset so it contains relationships between table and its caption.

In [30]:
# How many tables are inside a picture
import json
from collections import defaultdict

with open("data/COCO/train.json", "r", encoding="utf-8") as f:
    data = json.load(f)

images = {img["id"]: img for img in data["images"]}
categories = {cat["id"]: cat["name"] for cat in data["categories"]}
annotations = data["annotations"]

In [31]:
tables_per_document = defaultdict(list)

for ann in annotations:
    category_name = categories[ann["category_id"]]

    if category_name == "Table":
        img = images[ann["image_id"]]

        record = {
            "doc_name": img["doc_name"],
            "page_no": img["page_no"],
            "image_id": ann["image_id"],
            "bbox": ann["bbox"],
            "area": ann["area"],
        }

        tables_per_document[img["doc_name"]].append(record)


In [32]:
tables_len = 0
for doc, tables in tables_per_document.items():
    print(f"{doc}: {len(tables)} tabuliek")
    tables_len = tables_len + len(tables)

NYSE_F_2004.pdf: 93 tabuliek
NYSE_IP_2004.pdf: 103 tabuliek
NYSE_VNO_2003.pdf: 92 tabuliek
NYSE_RELX_2003.pdf: 152 tabuliek
NYSE_STZ_2004.pdf: 68 tabuliek
TSX_BBD_2004.pdf: 111 tabuliek
NYSE_BG_2004.pdf: 64 tabuliek
NYSE_TCB_2002.pdf: 84 tabuliek
OTC_FUJHY_2003.pdf: 61 tabuliek
TSX_BMO_2002.pdf: 127 tabuliek
OTC_FJTSF_2000.pdf: 40 tabuliek
NASDAQ_IBKC_2002.pdf: 54 tabuliek
NASDAQ_CWCO_2000.pdf: 33 tabuliek
NYSE_FC_2000.pdf: 55 tabuliek
ASX_SUL_2004.pdf: 47 tabuliek
TSX_FCR_2003.pdf: 64 tabuliek
TSX_WTE.UN_2002.pdf: 25 tabuliek
NASDAQ_BOKF_2001.pdf: 66 tabuliek
NYSE_AZO_2003.pdf: 34 tabuliek
NYSE_AZ_2003.pdf: 205 tabuliek
NYSE_EL_2003.pdf: 43 tabuliek
NYSE_OII_2003.pdf: 2 tabuliek
NASDAQ_STRT_2001.pdf: 26 tabuliek
NYSE_RDN_2001.pdf: 41 tabuliek
NYSE_DW_2003.pdf: 36 tabuliek
NYSE_ORH_2004.pdf: 55 tabuliek
NASDAQ_FRME_2003.pdf: 61 tabuliek
OTC_TOSYY_2003.pdf: 36 tabuliek
NYSE_GCI_2001.pdf: 75 tabuliek
NYSE_AA_2004.pdf: 68 tabuliek
ASX_SBM_2004.pdf: 64 tabuliek
NASDAQ_AGYS_2003.pdf: 44 tab

In [4]:
!pip install ultralytics

  Using cached polars-1.37.1-py3-none-any.whl.metadata (10 kB)
  Using cached ultralytics_thop-2.0.18-py3-none-any.whl.metadata (14 kB)
  Using cached polars_runtime_32-1.37.1-cp310-abi3-win_amd64.whl.metadata (1.5 kB)
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   -------- ------------------------------- 0.3/1.2 MB ? eta -:--:--
   -------------------------- ------------- 0.8/1.2 MB 2.2 MB/s eta 0:00:01
   ----------------------------------- ---- 1.0/1.2 MB 2.0 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 1.8 MB/s  0:00:00
Using cached polars-1.37.1-py3-none-any.whl (805 kB)
Using cached polars_runtime_32-1.37.1-cp310-abi3-win_amd64.whl (44.9 MB)
Using cached ultralytics_thop-2.0.18-py3-none-any.whl (28 kB)

   ---------------------------------------- 0/4 [polars-runtime-32]
   ---------------------------------------- 0/4 [polars-runtime-32]
   ---------------------------------------- 0/4 [polars-runtime-32]
   ----------------------

In [7]:
!pip install --user doclayout-yolo

In [1]:
import torch
from doclayout_yolo import YOLOv10
from doclayout_yolo.utils import ops

# --- ZAČIATOK OPRAVY (PATCH) ---
# Uložíme si pôvodnú funkciu, aby sme ju nezničili
original_nms = ops.non_max_suppression

def patched_nms(prediction, conf_thres=0.25, iou_thres=0.45, classes=None, agnostic=False, multi_label=False, labels=(), max_det=300, nc=0, max_time_img=0.05, max_nms=30000, max_wh=7680, in_place=True, rotated=False):
    """
    Táto funkcia skontroluje, či model nevrátil slovník (dict).
    Ak áno, vytiahne z neho kľúčovú časť 'one2one', ktorú potrebuje NMS.
    """
    if isinstance(prediction, dict):
        # DocLayout/YOLOv10 zvykne vracať dict s kľúčmi 'one2one' a 'one2many'
        if 'one2one' in prediction:
            prediction = prediction['one2one']
        else:
            # Ak nepoznáme kľúč, vezmeme prvú hodnotu (záchranná brzda)
            prediction = list(prediction.values())[0]
            
    # Zavoláme pôvodnú funkciu s opravenými dátami
    return original_nms(prediction, conf_thres, iou_thres, classes, agnostic, multi_label, labels, max_det, nc, max_time_img, max_nms, max_wh, in_place, rotated)

# Prepíšeme chybnú funkciu v knižnici našou opravenou verziou
ops.non_max_suppression = patched_nms
# --- KONIEC OPRAVY ---

# TERAZ UŽ TVOJ KÓD PÔJDE:
model = YOLOv10('doclayout_yolo_docstructbench_imgsz1024.pt')

results = model.predict(
    'data/PNG/fecf39d4d98045cc36c496707863aacb8214f60f527210b620c0eff708c4bbca.png', 
    save=True, 
    imgsz=1024,
    conf=0.25 # voliteľné: prah istoty
)

C:\Users\posko\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



image 1/1 D:\Bakalarka\gat\data\PNG\fecf39d4d98045cc36c496707863aacb8214f60f527210b620c0eff708c4bbca.png: 1024x1024 4 titles, 11 plain texts, 3 abandons, 1 table, 2 table_captions, 946.4ms
Speed: 7.1ms preprocess, 946.4ms inference, 2.0ms postprocess per image at shape (1, 3, 1024, 1024)
Results saved to runs\detect\predict
